# Analisis Sentimen - Naive Bayes + TF-IDF
Dataset: Review produk dengan label `0` = Negatif, `1` = Netral, `2` = Positif

## 1. Import Library

In [30]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB # ComplementNB lebih bagus untuk data tidak seimbang dan teks negatif
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from scipy.sparse import save_npz, load_npz
from google.colab import files

## 2. Upload & Load Dataset

In [31]:
# Upload file Excel (data-bersih-teratur.xlsx)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# PERBAIKAN: file Excel harus pakai read_excel, bukan read_csv
df = pd.read_excel(filename)

# Hapus baris dengan label NaN
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("DATASET")
print(f"Shape: {df.shape}")
print(df.head())
print("\nDistribusi Label:")
print(df['label'].value_counts().sort_index())

Saving data-bersih-teratur.xlsx to data-bersih-teratur (2).xlsx
DATASET
Shape: (68067, 5)
   review_id                                             review  rating  \
0        0.0                                      slow delivery     1.0   
1        1.0              Dateng goods do not conform pesanan??     1.0   
2        2.0                             PSN k its 20 other DTG     1.0   
3        3.0  I am expected that it have a frame and painted...     1.0   
4        4.0                   The product quality is not good.     1.0   

                                      review_cleaned  label  
0                                      slow delivery      0  
1                dateng goods do not conform pesanan      0  
2                                psn k its other dtg      0  
3  i am expected that it have a frame and painted...      0  
4                    the product quality is not good      0  

Distribusi Label:
label
0    11896
1    15528
2    40643
Name: count, dtype: int64


## 3. TF-IDF Vectorization

In [42]:
review = df['review_cleaned'].fillna('')
label = df['label']

tfidf = TfidfVectorizer(
    max_features=100000,       # naikkan agar kata negatif ikut tertangkap
    stop_words=None,         # HAPUS stop_words agar "not", "no", "never" tidak ikut dihapus
    ngram_range=(1, 2)       # tangkap frasa seperti "not good", "very disappointed"
)

# PERBAIKAN: X didefinisikan di sini supaya bisa dipakai di cell split
X = tfidf.fit_transform(review)

print(f"Shape matrix TF-IDF: {X.shape}")

# Simpan model dan matrix
joblib.dump(tfidf, 'model_tfidf.joblib')
save_npz('matrix_tfidf.npz', X)

print("Model dan matrix berhasil disimpan!")

Shape matrix TF-IDF: (68067, 100000)
Model dan matrix berhasil disimpan!


## 4. Split Data Training & Testing

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    label,
    test_size=0.2,
    random_state=42,
    stratify=label
)

print(f"Data Training : {X_train.shape[0]} baris")
print(f"Data Testing  : {X_test.shape[0]} baris")

Data Training : 54453 baris
Data Testing  : 13614 baris


## 5. Training Model Naive Bayes

In [44]:
model = ComplementNB()
model.fit(X_train, y_train)

print("Training model berhasil!")

Training model berhasil!


## 6. Evaluasi Model

In [45]:
prediksi = model.predict(X_test)

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, prediksi))

print("\nACCURACY")
print(accuracy_score(y_test, prediksi))

print("\nCLASSIFICATION REPORT")
# PERBAIKAN: target_names disesuaikan dengan 3 label yang ada
print(classification_report(y_test, prediksi, target_names=['Negatif', 'Netral', 'Positif']))

CONFUSION MATRIX
[[1912  258  209]
 [ 801 1384  921]
 [ 849  570 6710]]

ACCURACY
0.7349786983987072

CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Negatif       0.54      0.80      0.64      2379
      Netral       0.63      0.45      0.52      3106
     Positif       0.86      0.83      0.84      8129

    accuracy                           0.73     13614
   macro avg       0.67      0.69      0.67     13614
weighted avg       0.75      0.73      0.73     13614



## 7. Prediksi Review Baru

In [50]:
review_baru = [input("Masukkan review : ")]

review_vector = tfidf.transform(review_baru)
hasil = model.predict(review_vector)

# PERBAIKAN: label ada 3 (0, 1, 2) bukan hanya 2
label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

print(f"\nReview  : {review_baru[0]}")
print(f"Sentimen: {label_map[hasil[0]]}")

Masukkan review : this thing is suck 

Review  : this thing is suck 
Sentimen: Negatif


---
## (Opsional) Load Ulang Model Jika Session Baru

In [37]:
# Jalankan cell ini HANYA jika session Colab sudah expired / restart
# Upload ketiga file: data-bersih-teratur.xlsx, model_tfidf.joblib, matrix_tfidf.npz
uploaded = files.upload()

df      = pd.read_excel('data-bersih-teratur.xlsx')
df      = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

tfidf        = joblib.load('model_tfidf.joblib')
X            = load_npz('matrix_tfidf.npz')    # load_npz, bukan joblib.load
label        = df['label']

print("Model:", tfidf)
print("Matrix shape:", X.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

KeyboardInterrupt: 